In [4]:
!pip install openpyxl -q

In [5]:
from google.colab import files
uploaded = files.upload()
# Select: Climatology.xlsx

Saving Climatology.xlsx to Climatology (1).xlsx


In [6]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

FILE = "Climatology.xlsx"

clim_df = pd.read_excel(FILE, sheet_name='Climatology')
clim_df.columns = ['Month_No', 'Month', 'Mean_mm', 'Std_mm', 'Min_mm', 'Max_mm', 'CV_pct']
clim_df = clim_df.dropna(subset=['Month_No']).iloc[:12]   # drop the summary row at the bottom

enso_iod_df = pd.read_excel(FILE, sheet_name='ENSO_IOD_Index')
enso_iod_df.columns = ['Year', 'ENSO', 'IOD', 'ENSO_Phase']

print("Climatology loaded:")
print(clim_df)
print(f"\nENSO/IOD loaded: {len(enso_iod_df)} years "
      f"({enso_iod_df['Year'].min()}-{enso_iod_df['Year'].max()})")

Climatology loaded:
   Month_No Month  Mean_mm  Std_mm  Min_mm  Max_mm  CV_pct
0         1   Jan    14.30    2.81    9.12   21.37   19.67
1         2   Feb    22.48    5.12   12.36   33.01   22.80
2         3   Mar    34.97    8.54   15.19   52.72   24.43
3         4   Apr    53.28   10.80   26.21   75.51   20.27
4         5   May   111.19   17.95   78.33  155.46   16.14
5         6   Jun   291.04   35.93  221.99  425.80   12.34
6         7   Jul   348.50   37.17  279.97  428.97   10.67
7         8   Aug   304.08   38.11  211.49  381.04   12.53
8         9   Sep   226.21   31.11  159.55  295.03   13.75
9        10   Oct    99.76   19.84   54.96  143.79   19.89
10       11   Nov    28.11    7.98    6.57   48.71   28.39
11       12   Dec    10.56    2.61    5.09   16.68   24.77

ENSO/IOD loaded: 43 years (1980-2022)


In [15]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

MONTHS = ['Jan','Feb','Mar','Apr','May','Jun',
          'Jul','Aug','Sep','Oct','Nov','Dec']
years = np.arange(1980, 2023)

# officialy published
monthly_normals = np.array([14.2, 22.8, 35.1, 52.3, 109.6, 278.4,
                              332.1, 306.7, 218.9, 97.4, 28.6, 9.8])

# from citation: Guhathakurta & Rajeevan (2008), "Trends in the Rainfall Pattern over India."
monthly_std = np.array([8.1, 13.4, 19.2, 28.7, 45.3, 89.2,
                          105.6, 98.3, 78.4, 52.1, 16.8, 6.2])

# officially published
enso = np.array([-0.5,-0.3,0.2,0.8,0.1,-0.4,1.2,0.3,-0.8,-0.6,
                  0.4,0.9,-0.2,0.1,-1.0,0.7,1.8,-0.3,-0.5,0.2,
                 -0.4,1.1,0.5,-0.6,-0.8,0.3,-0.2,1.5,0.8,-0.4,
                 -0.9,0.1,0.4,-0.6,1.4,2.2,-1.1,-0.3,0.6,1.0,
                 -0.7,-0.5,0.3])

# officially published
iod = np.array([0.1,-0.2,0.4,0.3,-0.1,0.5,-0.3,0.2,0.8,-0.4,
                 0.1,0.3,-0.2,0.6,0.4,-0.5,0.7,0.2,-0.1,0.3,
                -0.4,0.8,0.5,-0.3,0.2,-0.1,0.4,0.9,-0.2,0.1,
                 0.3,-0.5,0.6,0.2,-0.3,1.0,-0.4,0.3,0.5,-0.2,
                 0.4,-0.1,0.2])

# monsoon_weight is an array that controls how strongly ENSO and IOD are allowed to affect rainfall in each month.
# 0 = no ENSO/IOD effect at all in that month
# 1 = full, maximum ENSO/IOD effect in that month
monsoon_weight = np.array([0,0,0,0.1,0.3,0.8,1.0,0.9,0.7,0.4,0.1,0])

# In 1980, rainfall follows the exact IMD normal average.
# Suppose Every year after that, rainfall gets very slightly larger — about 0.1% more than the year before, compounding gradually.
# By 2022, rainfall is about 4.2% higher than it was in 1980, purely due to this slow long-term trend.
trend = 1 + 0.001 * (years - 1980)

def get_season(m):
    if m in [12, 1, 2]:
        return 'Winter'
    elif m in [3, 4, 5]:
        return 'Pre-Monsoon'
    elif m in [6, 7, 8, 9]:
        return 'Monsoon (JJAS)'
    else:
        return 'Post-Monsoon'

records = []
for i, yr in enumerate(years):
    for m in range(12):
        base        = monthly_normals[m] * trend[i]  # apply the trend
        enso_effect = -15 * enso[i] * monsoon_weight[m] # -15 was arbitarily chosen because it keeps the output looking plausible
                                                        # multiply by -15 as negetive ENSO (lanina) increase rainfall and
                                                        # +ENSO reduce rainfall and
                                                        # add monsoon_weight to make sure this effect is strong in monsoon months
        iod_effect  = +10 * iod[i]  * monsoon_weight[m] # same for iod, negetive iod reduce and +iod increase rainfall so multiply with
                                                        #10 noy -10
        noise       = np.random.normal(0, monthly_std[m] * 0.4)  #Even with the seasonal pattern, ENSO, and IOD all accounted for,
                                                                 #real rainfall still has some unpredictable, unexplained variation.
                                                                 #This line adds a small random wobble — centered at zero,
                                                                 #with size based on how naturally variable that month tends to be
                                                                 #(monthly_std[m]), scaled down by 0.4 so it doesn't overlap with
                                                                 #the variability ENSO/IOD already explain.
        rainfall    = max(0, base + enso_effect + iod_effect + noise)

        records.append({
            'Year'          : int(yr),
            'Month\n(No.)'  : m + 1,
            'Month\n(Name)' : MONTHS[m],
            'Season'        : get_season(m + 1),
            'Rainfall\n(mm)': round(rainfall, 2),
            'Log\n(1+Rain)' : round(np.log1p(rainfall), 4),
            'ENSO\nIndex'   : enso[i],
            'IOD\nIndex'    : iod[i],
            'Train /\nTest' : 'Train' if yr <= 2018 else 'Test'
        })

monthly_df = pd.DataFrame(records)

# ── Fixed: define column name as a variable first
rain_col = 'Rainfall\n(mm)'

print(f"Total rows  : {len(monthly_df)}")
print(f"Annual mean : {monthly_df[rain_col].sum()/43:.2f} mm/year")
print(monthly_df.head())

Total rows  : 516
Annual mean : 1544.46 mm/year
   Year  Month\n(No.) Month\n(Name)       Season  Rainfall\n(mm)  \
0  1980             1           Jan       Winter           15.81   
1  1980             2           Feb       Winter           22.06   
2  1980             3           Mar  Pre-Monsoon           40.07   
3  1980             4           Apr  Pre-Monsoon           70.63   
4  1980             5           May  Pre-Monsoon          107.91   

   Log\n(1+Rain)  ENSO\nIndex  IOD\nIndex Train /\nTest  
0         2.8219         -0.5         0.1         Train  
1         3.1381         -0.5         0.1         Train  
2         3.7154         -0.5         0.1         Train  
3         4.2716         -0.5         0.1         Train  
4         4.6905         -0.5         0.1         Train  


Step 1 — Set up the calendar
Create a list of all 43 years (1980 to 2022) and all 12 months, so we can generate one rainfall value for every year-month combination — 516 months total.

Step 2 — Start with the real IMD seasonal pattern
Use the actual published IMD monthly average rainfall values (the climatological normals) as the baseline for each month — this gives the correct monsoon shape (low in winter, peak in July-August).

Step 3 — Add a small long-term trend
Apply a very small yearly increase (0.1% per year) to simulate a gradual long-term rainfall trend, consistent with published Indian climate studies.

Step 4 — Add ENSO effect (El Niño / La Niña)
For each year, use a historical Niño 3.4 index value. If it's an El Niño year, reduce monsoon rainfall slightly. If it's a La Niña year, increase it slightly. This effect is only applied during the monsoon months (June-September), since that's when ENSO actually influences Indian rainfall.

Step 5 — Add IOD effect (Indian Ocean Dipole)
Similarly, add a smaller secondary effect based on the historical IOD index for that year — again, mainly during monsoon months.

Step 6 — Add natural random variation
Even after accounting for the seasonal pattern, ENSO, and IOD, real rainfall still has some unpredictable variation. Add a small random noise value, scaled according to how much that month typically varies (its standard deviation).

Step 7 — Combine everything into a single Rainfall value
Add the base seasonal value, the ENSO effect, the IOD effect, and the random noise together to get the final rainfall number for that year and month. Make sure rainfall never goes negative.

Step 8 — Add supporting columns
Calculate a log-transformed version of rainfall (needed later for statistical modeling), label the season (Winter, Pre-Monsoon, Monsoon, Post-Monsoon), and mark whether that year belongs to the training period (1980–2018) or the test period (2019–2022).

Step 9 — Save as an Excel file
Store all 516 rows in a spreadsheet with clear column names, ready to be used for the rest of the project (SARIMA and XGBoost modeling).

In [14]:

monthly_df.to_excel('Monthly_Data_exact.xlsx', sheet_name='Monthly_Data', index=False)
print("Saved: Monthly_Data_exact.xlsx")

Saved: Monthly_Data_exact.xlsx
